# Machine Learning on text with TF-IDF and LSA

In [ ]:
!pip install --upgrade 'imbalanced-learn'
import imblearn
from IPython.display import display
import matplotlib.pyplot as plt
import numpy
import pandas
import pathlib
import seaborn
import sklearn.decomposition
import sklearn.ensemble
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline
import sklearn.svm

## Getting data
Execute the cell below to retrieve the two corpora of this lab.

The first one is a collection of movie plots and other metadata such as the release year, the director or the genre.

The second one, 20 Newsgroups, gathers the messages of 20 discussion forums: it will be used for the latent semantic analysis.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wikipedia-movie-plots
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
movie_plot_path = "wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Loading the data in a pandas dataframe

Load the csv file in a dataframe.

Check that the columns are consistent with the csv file. How many movies are there in this dataset?

In [ ]:
# Your code here

### Solution

In [ ]:
df = pandas.read_csv(movie_plot_path)
print(f"Columns: {', '.join(df.columns)}")
print(f"Shape: {df.shape}")

## Selecting data

We will try to predict the genre of a movie using its plot. To do so, we will limit ourselves to the 5 most present genres in the dataset. We will also remove the movies with the most present genre: `unknown`, for obvious reasons.

What are the genres that we will keep?

Build a new dataframe containing only the movies of those genres.

How many movies are left?

Useful functions:

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)

In [ ]:
# Your code here

### Solution

In [ ]:
counts = df["Genre"].value_counts().drop("unknown")
genres = counts.index.values[:5]
print(f"Selected genres: {', '.join(genres)}")
print(f"Number of selected movies: {counts[genres].sum()}")

In [ ]:
clean_df = df[df["Genre"].isin(genres)]
seaborn.countplot(clean_df, x="Genre")

## TF-IDF representation

Use [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to convert plot texts to vectors.

Remove stop words and keep only the 3000 most frequent words.

In [ ]:
x = clean_df["Plot"]

# Your code here

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000
)
X = vectorizer.fit_transform(x)
feature_names = vectorizer.get_feature_names_out()
print(f"First 10 columns:  {', '.join(feature_names[:10])}")

In [ ]:
indices_0 = X[0].indices
words_0 = [vectorizer.get_feature_names_out()[i] for i in X[0].indices]
print(f"Columns > 0 in the first row of X: {indices_0}")
print(f"Corresponding words: {', '.join(words_0)}")
print(f"First value of x: {x.iloc[0]}")

## A corpus for LSA

Latent semantic analysis (LSA) is a truncated SVD of the documents × words matrix: each axis groups words that appear together. For it to show anything, the corpus needs well-marked themes. Movie plots don't have them: their axes fill up with first names (`joe`, `jack`, `mary`). So we switch to 20 Newsgroups, whose messages come from 20 thematic forums: sports, computing, religion, politics…

Execute the cell below. It loads the 11,314 messages of the corpus's training part, and removes their header (sender, subject…), which stops at the first blank line.

In [ ]:
# Each folder is a forum, then one message per file
messages = [
  p.read_text(encoding="latin-1").partition("\n\n")[2]
  for p in sorted(pathlib.Path("20news-bydate-train").glob("*/*"))
]

print(f"{len(messages)} messages retrieved")
print(messages[0][:300])

## TF-IDF representation of the messages

As for the movies, convert `messages` to vectors with `TfidfVectorizer`: remove the stop words and keep the 3000 most frequent words. Also remove the words present in more than 30% of the messages, or in fewer than 5 messages.

Display the size of the matrix and the average number of non-zero values per row: how many of the 3000 words does a message use?

**Useful documentation**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Your code here

### Additional hints

`stop_words` and `max_features` work as they did for the movies. `max_df=0.3` removes the words present in more than 30% of the documents (a float is a share of the documents) and `min_df=5` those present in fewer than 5 documents (an integer is a number of documents).

`X_messages.shape` gives the size of the matrix and `X_messages.nnz` its number of non-zero values. You should get `(11314, 3000)` and about 55 words per message.

```python
message_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=..., max_df=..., min_df=...
)
X_messages = message_vectorizer.fit_transform(...)
feature_names_messages = message_vectorizer.get_feature_names_out()
print(f"Matrix size: {X_messages.shape}")
print(f"Words per message: {...:.0f}")
```

### Solution

In [ ]:
message_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000, max_df=0.3, min_df=5
)
X_messages = message_vectorizer.fit_transform(messages)
feature_names_messages = message_vectorizer.get_feature_names_out()
print(f"Matrix size: {X_messages.shape}")
print(f"Words per message: {X_messages.nnz / X_messages.shape[0]:.0f}")

## Latent Semantic Analysis

Use `TruncatedSVD` to compute an LSA with 6 axes on `X_messages`.

Display the singular value of each axis, the share of variance it keeps, and the share the 6 axes keep together. What share of the variance do they keep? Does the axis with the largest singular value keep the largest variance?

**Useful documentation**

- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
# Your code here

### Additional hints

`n_components=6` sets the number of axes, `n_iter=100` refines the computation and `random_state=42` gives the same axes from one run to the next. After `svd.fit(X_messages)`, the attributes `singular_values_` and `explained_variance_ratio_` hold one value per axis.

You should get a singular value of 16 for axis 1, and about 2.8% of the variance for the 6 axes.

```python
svd = sklearn.decomposition.TruncatedSVD(...)
svd.fit(...)

print(f"Singular values: {svd.singular_values_.round(2)}")
print(f"Share of variance per axis: {...}")
print(f"Share of variance of the 6 axes: {...:.1%}")
```

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=6, n_iter=100, random_state=42)
svd.fit(X_messages)

print(f"Singular values: {svd.singular_values_.round(2)}")
print(f"Share of variance per axis: {svd.explained_variance_ratio_.round(4)}")
print(f"Share of variance of the 6 axes: {svd.explained_variance_ratio_.sum():.1%}")

## Important words

The 6 axes keep less than 3% of the variance: that's normal, and it isn't a failure. A message uses only about 55 of the 3000 words, and most of the variance comes from these choices, specific to each message: it would take about 500 axes to keep half of it. LSA retains only what many messages share: judge an axis by its words, not by its share of variance.

Axis 1 isn't a theme. `TruncatedSVD` doesn't center the matrix, none of whose values is negative: this axis has only positive coefficients, and it gives the corpus's average vocabulary. That's how it has the largest singular value without keeping the largest variance. Themes are read from axis 2 on.

The sign of a coefficient carries the meaning: an axis opposes the pole of its positive coefficients to the pole of its negative ones. For each axis, display the 8 most positive words and the 8 most negative words. Then, for axes 2 to 6, name each pole: what theme is it about?

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Your code here

### Additional hints

`svd.components_` has one row per axis and one column per word. `numpy.argsort(svd.components_, axis=1)` sorts each row: for an axis, `indices` goes from the most negative coefficient to the most positive one. So `indices[:8]` gives the 8 most negative ones, `indices[::-1][:8]` the 8 most positive ones, and `feature_names_messages[indices[:8]]` the corresponding words.

You should find `god` and `jesus` at one end of axis 2, `windows` and `card` at the other. Only the opposition matters: which pole is the positive one depends only on the computation.

```python
n_words = 8

for number, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positive = ...
  negative = ...
  print(f"Axis {number}")
  print(f"  + {', '.join(positive)}")
  print(f"  - {', '.join(negative)}")
```

### Solution

In [ ]:
n_words = 8

# argsort sorts each axis's coefficients, from the most negative to the most positive
for number, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positive = feature_names_messages[indices[::-1][:n_words]]
  negative = feature_names_messages[indices[:n_words]]
  print(f"Axis {number}")
  print(f"  + {', '.join(positive)}")
  print(f"  - {', '.join(negative)}")

# Axis 1: the average vocabulary. Its coefficients are all positive: the words shown
# as negative are just the closest to 0.
# Axis 2: religion against computing
# Axis 3: religion against sports
# Axis 4: cryptography against religion and sports
# Axis 5: hardware against software
# Axis 6: cryptography against the Middle East
# The poles aren't pure (`windows` on the religion side of axis 3): the axes have to
# stay orthogonal, so a theme spreads over several axes.

## Supervised Approach

Back to the movies: we'll try to predict the movie's genre from its plot.

To do so, start by extracting the genre of each movie into a pandas series `y`. scikit-learn's classifiers take these strings as they are.

`drama` has 6 times as many movies as `thriller`. What accuracy does a model that always answers `drama` get? `value_counts(normalize=True)` gives the share of each genre. That's the score to beat.

In [ ]:
# Your code here

### Solution

In [ ]:
y = clean_df["Genre"]

shares = y.value_counts(normalize=True)
display(shares)

print(f"Accuracy when always answering drama: {shares['drama']:.1%}")

## Training

Use a classification algorithm of your choice to train a model that takes the tf-idf vectors of the plots as input and predicts the right class.

Evaluate it with cross-validation ([`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)). The vocabulary and the idf weights are part of the model: so that they're only learned on each fold's training data, put the `TfidfVectorizer` and the classifier in a single pipeline ([`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)), applied to the texts `x`. The file is sorted by the movies' origin: so that the folds don't follow these blocks, shuffle them with [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).

Display the accuracy, the micro and macro precisions, and the corresponding confusion matrix.

In [ ]:
# Your code here

### Solution

In [ ]:
def pipeline(model):
  # The vectorizer is part of the pipeline: on each cross-validation fold, it
  # only learns its vocabulary and idf weights from the fold's training data
  model = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    model,
  )
  # The file is sorted by origin: shuffle so that the folds don't follow these blocks
  folds = sklearn.model_selection.StratifiedKFold(5, shuffle=True, random_state=0)
  predictions = sklearn.model_selection.cross_val_predict(model, x, y, cv=folds)

  # Accuracy is the number of correct predictions over the number of predictions
  accuracy = sklearn.metrics.accuracy_score(y, predictions)

  # Micro precision pools the true and false positives of every class before
  # dividing: when each example has a single class, it equals the accuracy
  micro_precision = sklearn.metrics.precision_score(y, predictions, average="micro")

  # Macro precision is the unweighted mean of each class's precision: small
  # classes count as much as large ones
  macro_precision = sklearn.metrics.precision_score(y, predictions, average="macro")

  print(
    f"Accuracy: {accuracy:.2f}, "
    f"micro precision: {micro_precision:.2f}, "
    f"macro precision: {macro_precision:.2f}"
  )

  confusion_matrix = sklearn.metrics.confusion_matrix(y, predictions, normalize="true")
  seaborn.heatmap(
    confusion_matrix,
    vmin=0,
    vmax=1,
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap=seaborn.color_palette("Blues", as_cmap=True),
  )
  plt.plot()

In [ ]:
pipeline(sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200))

In [ ]:
# With several bootstrapped logistic regressions with under-sampling
pipeline(
  imblearn.ensemble.BalancedBaggingClassifier(
    sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200)
  )
)

In [ ]:
# With over-sampling of least represented genres
pipeline(
  imblearn.pipeline.make_pipeline(
    imblearn.over_sampling.RandomOverSampler(),
    sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200),
  )
)

In [ ]:
# With a fast SVM
# pipeline(sklearn.svm.LinearSVC(class_weight="balanced"))

In [ ]:
# With a random forest
# pipeline(sklearn.ensemble.RandomForestClassifier(n_estimators=100,
#                                                  class_weight="balanced"))

In [ ]:
# With a slow SVM but with a powerful kernel
# pipeline(sklearn.svm.SVC(class_weight="balanced"))